# Python (OOP) - dziedziczenie (ang. *inheritance*)

_Mikołaj Leszczuk_

![Ilustracja do zajęć o dziedziczeniu](https://pythoncodeshark.wordpress.com/wp-content/uploads/2014/01/inheritance-meem.jpg)
![](https://i.creativecommons.org/l/by/4.0/88x31.png)

# Podstawy

Klasa pochodna przejmuje dostęp do atrybutów i metod klasy bazowej. Może dodać własne lub zmienić zachowanie odziedziczonej metody. Na rysunku jedna klasa pochodna ma jedną klasę bazową.

![Dziedziczenie pojedyncze: jedna klasa bazowa](https://upload.wikimedia.org/wikipedia/commons/3/32/Single_Inheritance.jpg)

Dziedziczenie definiowane jest za pomocą składni:

In [1]:
class KlasaBazowa:
    pass

class KlasaPochodna(KlasaBazowa):
    pass

Wracamy do wersji `KontoBankowe` z pokazu w notebooku 11, sprzed ćwiczenia z przelewem. Najpierw utworzymy klasę pochodną bez własnych metod.

In [2]:
class KontoBankowe:
    def __init__(self, nazwa, stan=0):
        self.nazwa = nazwa
        self.stan = stan

    def info(self):
        print("nazwa:", self.nazwa)
        print("stan:", self.stan)

    def wyplac(self, ilosc):
        self.stan -= ilosc

    def wplac(self, ilosc):
        self.stan += ilosc 


class KontoDebetowe(KontoBankowe):
    pass

#### Co już odziedziczyło konto debetowe?

In [ ]:
konto = KontoDebetowe("Jan Nowak", 100)
konto.wplac(50)
konto.info()

## Super

`super()` sięga do kolejnej implementacji metody w ustalonej przez Pythona kolejności (MRO). Tutaj jest to implementacja z `KontoBankowe`. Po zdefiniowaniu własnego `__init__()` klasa pochodna musi sama wywołać inicjalizację bazową, jeżeli chce z niej skorzystać.

#### Bezpośrednie wywołanie

```python
def __init__(self, nazwa, stan=0, limit=0):
    KontoBankowe.__init__(self, nazwa, stan)
    self.limit = limit
```

#### Wywołanie przez `super()`

```python
def __init__(self, nazwa, stan=0, limit=0):
    super().__init__(nazwa, stan)
    self.limit = limit
```

#### Przesłanianie metody i wywołanie wersji bazowej

Nowe `__init__()` ustawia limit, a nowe `wyplac()` rozstrzyga, czy wypłata się mieści w limicie. Gdy jest dozwolona, `super().wyplac()` wykonuje odejmowanie z klasy bazowej.

In [12]:
class KontoDebetowe(KontoBankowe):
    def __init__(self, nazwa, stan=0, limit=0):
        super().__init__(nazwa, stan)
        self.limit = limit

    def wyplac(self, ilosc):
        """Jeżeli stan konta po operacji przekroczyłby limit, przerwij."""
        if (self.stan - ilosc) < (-self.limit):
            print("Brak srodkow na koncie")
        else:
            super().wyplac(ilosc)

In [13]:
account = KontoDebetowe("Jan Nowak", 0, 1000)

In [14]:
account.info()

nazwa: Jan Nowak
stan: 0


In [15]:
account.wplac(500)

In [16]:
account.info()

nazwa: Jan Nowak
stan: 500


In [17]:
account.wyplac(1000)

In [18]:
account.info()

nazwa: Jan Nowak
stan: -500


In [19]:
account.wyplac(1000)

Brak srodkow na koncie


In [20]:
account.info()

nazwa: Jan Nowak
stan: -500


### Ćwiczenie: wypłata na granicy limitu

Utwórz konto z saldem 100 i limitem 300. Wypłać 400, a potem spróbuj wypłacić jeszcze 1. Sprawdź saldo po każdej próbie i wyjaśnij, skąd wziął się wynik.

#### Rozwiązanie: saldo nie spada poniżej -300

In [ ]:
konto_testowe = KontoDebetowe("Jan Nowak", 100, 300)
konto_testowe.wyplac(400)
print(konto_testowe.stan)
assert konto_testowe.stan == -300

konto_testowe.wyplac(1)
print(konto_testowe.stan)
assert konto_testowe.stan == -300

## Ćwiczenia

### Utwórz podrzędną klasę `Bus`, która odziedziczy wszystkie zmienne i metody klasy `Vehicle`

#### Ćwiczenie

Poniżej jest gotowa klasa `Vehicle`. Dopisz `class Bus(Vehicle)` z pustym ciałem (`pass`). Utwórz autobus, podając dane do odziedziczonego `__init__()`, i wyświetl jego trzy atrybuty.

In [28]:
class Vehicle:

    def __init__(self, name, max_speed, mileage):
        self.name = name
        self.max_speed = max_speed
        self.mileage = mileage

Oczekiwany wynik. W rozwiązaniu zapis `f"{liczba:,}"` wstawia przecinek między grupy cyfr.

```
Nazwa pojazdu: Szkolne Volvo Prędkość: 180 Przebieg: 12,000
```

#### Rozwiązanie

In [29]:
class Vehicle:

    def __init__(self, name, max_speed, mileage):
        self.name = name
        self.max_speed = max_speed
        self.mileage = mileage

class Bus(Vehicle):
    pass

In [30]:
school_bus = Bus("Szkolne Volvo", 180, 12_000)

In [31]:
print(
    "Nazwa pojazdu:", school_bus.name,
    "Prędkość:", school_bus.max_speed,
    "Przebieg:", f"{school_bus.mileage:,}"
)

Nazwa pojazdu: Szkolne Volvo Prędkość: 180 Przebieg: 12,000


### Wspólny domyślny kolor pojazdów

#### Ćwiczenie

Dodaj do `Vehicle` atrybut klasy `color = "Biały"`. Sprawdź, czy odziedziczą go zarówno autobus, jak i samochód. Nazwa, prędkość i przebieg mają nadal pochodzić z konkretnej instancji.

In [32]:
class Vehicle:

    def __init__(self, name, max_speed, mileage):
        self.name = name
        self.max_speed = max_speed
        self.mileage = mileage

class Bus(Vehicle):
    pass

class Car(Vehicle):
    pass

Oczekiwany wynik:

```
Biały Szkolne Volvo Prędkość: 180 Przebieg: 12000
Biały Audi Q5 Prędkość: 240 Przebieg: 18000
```

#### Rozwiązanie

Atrybuty `name`, `max_speed` i `mileage` powstają osobno dla każdego pojazdu w `__init__()`. `color` jest wspólną wartością domyślną z klasy bazowej. Instancja może ją przesłonić własną wartością.

In [ ]:
class Vehicle:
    # Atrybut klasy
    color = "Biały"

    def __init__(self, name, max_speed, mileage):
        self.name = name
        self.max_speed = max_speed
        self.mileage = mileage

class Bus(Vehicle):
    pass

class Car(Vehicle):
    pass

In [33]:
school_bus = Bus("Szkolne Volvo", 180, 12_000)
print(
    school_bus.color, school_bus.name,
    "Prędkość:", school_bus.max_speed, "Przebieg:", school_bus.mileage
)

car = Car("Audi Q5", 240, 18_000)
print(
    car.color, car.name,
    "Prędkość:", car.max_speed, "Przebieg:", car.mileage
)

Biały Szkolne Volvo Prędkość: 180 Przebieg: 12000
Biały Audi Q5 Prędkość: 240 Przebieg: 18000


### Dziedziczenie klas

#### Ćwiczenie

Utwórz klasę `Bus`, która dziedziczy po klasie `Vehicle`. Podaj argument pojemności w metodzie `Bus.seating_capacity()` o domyślnej wartości `50`.

Poniższa klasa bazowa wymaga argumentu `capacity`. W `Bus` przesłoń metodę tak, żeby można ją było wywołać bez argumentu, ale nadal można było podać inną pojemność.

In [38]:
class Vehicle:
    def __init__(self, name, max_speed, mileage):
        self.name = name
        self.max_speed = max_speed
        self.mileage = mileage

    def seating_capacity(self, capacity):
        return f"Liczba miejsc siedzących w {self.name} to {capacity} pasażerów"

In [39]:
school_vehicle = Vehicle("Szkolne Volvo", 180, 12_000)

In [40]:
print(school_vehicle.seating_capacity(100))

Liczba miejsc siedzących w Szkolne Volvo to 100 pasażerów


In [ ]:
try:
    print(school_vehicle.seating_capacity())  # Brak domyślnej wartości
except TypeError as blad:
    print("Bez argumentu klasa bazowa zgłasza TypeError:", blad)

Oczekiwany wynik:

```
Liczba miejsc siedzących w Szkolne Volvo to 50 pasażerów
```

#### Rozwiązanie

In [42]:
class Vehicle:
    def __init__(self, name, max_speed, mileage):
        self.name = name
        self.max_speed = max_speed
        self.mileage = mileage

    def seating_capacity(self, capacity):
        return f"Liczba miejsc siedzących w {self.name} to {capacity} pasażerów"

class Bus(Vehicle):
    def seating_capacity(self, capacity=50):
        return super().seating_capacity(capacity)

In [43]:
school_bus = Bus("Szkolne Volvo", 180, 12_000)

In [44]:
print(school_bus.seating_capacity())

Liczba miejsc siedzących w Szkolne Volvo to 50 pasażerów


### Opłata za przejazd autobusem

#### Ćwiczenie

Utwórz klasę `Bus` dziedziczącą po `Vehicle`. Bazowa metoda `fare()` zwraca liczbę miejsc razy 100. Kod startowy dla autobusu z 50 miejscami wypisze 5000.

Przesłoń `fare()` w `Bus`: wywołaj `super().fare()` i dolicz do wyniku 10% na utrzymanie autobusu. Oczekiwany wynik: 5500.

In [45]:
class Vehicle:
    def __init__(self, name, mileage, capacity):
        self.name = name
        self.mileage = mileage
        self.capacity = capacity

    def fare(self):
        return self.capacity * 100

class Bus(Vehicle):
    pass

school_bus = Bus("Szkolne Volvo", 12, 50)
print("Całkowita opłata za przejazd autobusem wynosi:", school_bus.fare())

Całkowita opłata za przejazd autobusem wynosi: 5000


Oczekiwany wynik:

```
Całkowita opłata za przejazd autobusem wynosi: 5500
```

#### Rozwiązanie

In [46]:
class Vehicle:
    def __init__(self, name, mileage, capacity):
        self.name = name
        self.mileage = mileage
        self.capacity = capacity

    def fare(self):
        return self.capacity * 100

class Bus(Vehicle):
    def fare(self):
        amount = super().fare()
        amount += amount * 10 / 100
        return round(amount)

school_bus = Bus("Szkolne Volvo", 12, 50)
print("Całkowita opłata za przejazd autobusem wynosi:", school_bus.fare())

Całkowita opłata za przejazd autobusem wynosi: 5500


### Figury geometryczne

Stwórz klasę bazową dla figur geometrycznych:

In [1]:
import math


class Figura:
    def obwod(self):  # L/O
        """Obliczanie obwodu."""
        print("Obwód: ")

    def pole(self):  # S/P
        """Obliczanie pola powierzchni."""
        print("Pole: ")

In [2]:
f = Figura()

In [3]:
f.obwod()

Obwód: 


In [4]:
f.pole()

Pole: 


Zaimplementujemy teraz `Kolo(Figura)`: zachowaj nazwy metod `obwod()` i `pole()`, ale niech zwracają liczby obliczone z promienia. Po kole użyjemy tej samej klasy bazowej do zbudowania prostokąta.

Stała matematyczna π = 3,141592…, z dostępną dokładnością:

In [5]:
print(math.pi)

3.141592653589793


#### Koło

##### Ćwiczenie

Dla promienia `r` obwód wynosi `2 * math.pi * r`, a pole `math.pi * r ** 2`. Dodaj `__init__`, przechowaj promień i przesłoń obie metody. W każdej możesz zachować wypisywanie etykiety przez `super()`.

##### Rozwiązanie

In [6]:
class Kolo(Figura):
    def __init__(self, r):
        self.r = r

    def obwod(self):
        super().obwod()
        return 2 * math.pi * self.r

    def pole(self):
        super().pole()
        return math.pi * self.r ** 2

In [7]:
f1 = Kolo(5)

In [8]:
print(f1.obwod())

Obwód: 
31.41592653589793


In [9]:
print(f1.pole())

Pole: 
78.53981633974483


#### Prostokąt

##### Ćwiczenie

Utwórz klasę `Prostokat(Figura)`. Zapisz długość i szerokość, a następnie przesłoń `pole()` i `obwod()`.

![](https://upload.wikimedia.org/wikipedia/commons/9/96/Prostokat-rectangle.svg)
![](https://wikimedia.org/api/rest_v1/media/math/render/svg/570b1905a72d91a6a30a4b96b59cb61cb2e83ce5)
![](https://wikimedia.org/api/rest_v1/media/math/render/svg/b55ea3625c718c1700fda4b390485947f29934c2)

##### Rozwiązanie: prostokąt jako podklasa `Figura`

In [26]:
class Prostokat(Figura):
    def __init__(self, a, b):
        self.a, self.b = a, b

    def obwod(self):
        super().obwod()
        return 2 * (self.a + self.b)

    def pole(self):
        super().pole()
        return self.a * self.b

In [27]:
f5 = Prostokat(2, 5)

In [28]:
print(f5.obwod())

Obwód: 
14


In [29]:
print(f5.pole())

Pole: 
10


#### Kwadrat

##### Ćwiczenie

Utwórz klasę `Kwadrat(Prostokat)`, której podajesz tylko jeden bok. Użyj `super().__init__()`, żeby przekazać ten bok jako oba wymiary. Nie pisz ponownie metod liczących pole i obwód.

![](https://upload.wikimedia.org/wikipedia/commons/1/1a/In_square.png)
![](https://wikimedia.org/api/rest_v1/media/math/render/svg/22ba22f961bb6821bbfedec682f6b4d862b3eddb)
![](https://wikimedia.org/api/rest_v1/media/math/render/svg/939ffe659fc6dc88667915213a2a5e23ce27c84f)

##### Rozwiązanie

In [34]:
class Kwadrat(Prostokat):
    def __init__(self, a):
        super().__init__(a, a)

In [35]:
f6 = Kwadrat(5)

In [36]:
print(f6.obwod())

Obwód: 
20


In [37]:
print(f6.pole())

Pole: 
25


### Dokładny typ i klasa bazowa

`type(obiekt)` zwraca dokładną klasę obiektu. `isinstance(obiekt, Klasa)` sprawdza, czy obiekt jest instancją tej klasy lub jej podklasy.

#### Ćwiczenie

W poniższym przykładzie `Bus` dziedziczy po `Vehicle`. Zanim uruchomisz kod, przewidź wyniki:

1. Jaką klasę zwróci `type(school_bus)`?
2. Czy `isinstance(school_bus, Vehicle)` da `True`?

Dopisz oba sprawdzenia do komórki i ją uruchom.

In [34]:
class Vehicle:
    def __init__(self, name, mileage, capacity):
        self.name = name
        self.mileage = mileage
        self.capacity = capacity

class Bus(Vehicle):
    pass

school_bus = Bus("Szkolne Volvo", 12, 50)

#### Rozwiązanie

In [35]:
class Vehicle:
    def __init__(self, name, mileage, capacity):
        self.name = name
        self.mileage = mileage
        self.capacity = capacity

class Bus(Vehicle):
    pass

school_bus = Bus("Szkolne Volvo", 12, 50)

# użyj wbudowanej funkcji type() w Pythonie
print(type(school_bus))
print(isinstance(school_bus, Vehicle))

<class '__main__.Bus'>
True


##### Ćwiczenie: dokładny typ kwadratu

Dla kwadratu `f6` przewidź wyniki dwóch sprawdzeń:

1. `type(f6) is Prostokat`
2. `isinstance(f6, Prostokat)`

Uruchom oba wyrażenia. Uzasadnij, skąd bierze się różnica między wynikami.

##### Rozwiązanie: dokładny typ a klasa bazowa

In [ ]:
print(type(f6))
print(type(f6) is Prostokat)
print(isinstance(f6, Prostokat))

Pierwsze porównanie daje `False`: dokładna klasa `f6` to `Kwadrat`, a nie `Prostokat`. Drugie daje `True`, bo `Kwadrat` dziedziczy po `Prostokat`, a `isinstance()` uwzględnia podklasy.

# Zagadnienia uzupełniające

## Konto debetowe: jawne wywołanie klasy bazowej

Poniżej wracamy do przykładu z podstaw. Tym razem zamiast `super()` wskazujemy klasę `KontoBankowe` wprost. Można porównać, czy oba warianty dają taki sam wynik.

Rozszerzenie zachowania klasy `KontoBankowe`:

In [3]:
class KontoDebetowe(KontoBankowe):
    def __init__(self, nazwa, stan=0, limit=0):
        KontoBankowe.__init__(self, nazwa, stan)
        self.limit = limit

    def wyplac(self, ilosc):
        """Jeżeli stan konta po operacji przekroczyłby limit, przerwij."""
        if (self.stan - ilosc) < (-self.limit):
            print("Brak srodkow na koncie")
        else:
            KontoBankowe.wyplac(self, ilosc)

In [4]:
account = KontoDebetowe("Jan Nowak", 0, 1000)

In [5]:
account.info()

nazwa: Jan Nowak
stan: 0


In [6]:
account.wplac(500)

In [7]:
account.info()

nazwa: Jan Nowak
stan: 500


In [8]:
account.wyplac(1000)

In [9]:
account.info()

nazwa: Jan Nowak
stan: -500


In [10]:
account.wyplac(1000)

Brak srodkow na koncie


In [11]:
account.info()

nazwa: Jan Nowak
stan: -500


## Wielokrotne dziedziczenie

Jedna klasa pochodna ma więcej niż jedną bezpośrednią klasę bazową.

![Dziedziczenie wielokrotne: dwie bezpośrednie klasy bazowe](https://upload.wikimedia.org/wikipedia/commons/6/69/Multiple_Inheritance.jpg)

#### Dla porównania: dziedziczenie wielopoziomowe

Tu klasy tworzą łańcuch: klasa pochodna jednej klasy sama staje się bazą dla następnej.

![Dziedziczenie wielopoziomowe: łańcuch klas](https://upload.wikimedia.org/wikipedia/commons/0/0e/Multilevel_Inheritance.jpg)

<img src="https://img.devrant.io/devrant/rant/r_403541_YuG5k.jpg" alt="Żart o dziedziczeniu wielokrotnym" width="440">

#### Dwie klasy bazowe i `super()`

W `Pochodna(B, A)` wywołania `super().__init__()` idą od `Pochodna` przez `B` i `A` do `object`. Dzięki temu w tej instancji oba inicjalizatory zapiszą swoje atrybuty.

In [21]:
class A:
    """Rodzic pierwszy"""

    def __init__(self):
        super().__init__()
        self.a = "A"

    def fa(self):
        print("a:", self.a)


class B:
    """Rodzic drugi"""

    def __init__(self):
        super().__init__()
        self.b = "B"

    def fb(self):
        print("b:", self.b)


class Pochodna(B, A):
    """Dziecko"""

    def __init__(self):
        super().__init__()

`Klasa.__doc__` zwraca docstring tej klasy lub `None`, gdy klasa nie ma własnego. Nie zwraca docstringa klasy bazowej.

In [22]:
print(A.__doc__)
print(B.__doc__)
print(Pochodna.__doc__)

Rodzic pierwszy
Rodzic drugi
Dziecko


Działanie:

In [23]:
d = Pochodna()

In [24]:
print(d.a)

A


In [25]:
print(d.b)

B


In [26]:
d.fa()

a: A


In [27]:
d.fb()

b: B


Instancja `d` ma atrybuty `a` i `b`, ustawione przez inicjalizatory obu klas bazowych. Metody `fa()` i `fb()` są dostępne przez dziedziczenie.

#### Sprawdzenie kolejności klas (MRO)

In [ ]:
print([klasa.__name__ for klasa in Pochodna.__mro__])

## Dalsze zadania z figur

Użyj `Figura` z podstaw. W każdej podklasie zapisz wymiary w `__init__()` i uzupełnij `obwod()` oraz `pole()` albo odziedzicz metodę, która już daje właściwy wynik. Rozwiązania wariantów zależnych od wcześniejszych figur są ułożone po kolei.

#### Schemat i wzory dla koła

![](https://upload.wikimedia.org/wikipedia/commons/0/03/Circle-withsegments.svg)

![](https://wikimedia.org/api/rest_v1/media/math/render/svg/538bf78a77ecec3f84eddb43beab486a259cd87b)
![](https://wikimedia.org/api/rest_v1/media/math/render/svg/9f5a77550dc6995f6f8557e85720f2fda1d55d52)

#### Trójkąt równoboczny

##### Ćwiczenie

Dane: bok `a`. Obwód: `3 * a`, pole: `math.sqrt(3) * a ** 2 / 4`.

![](https://upload.wikimedia.org/wikipedia/commons/f/fc/Equilateral_Triangle.svg)
![](https://wikimedia.org/api/rest_v1/media/math/render/svg/dd90b00dbe4d139f034b10dfe180eb52e51302cf)
![](https://wikimedia.org/api/rest_v1/media/math/render/svg/008fa2bf38bf91f153b215b2a25256765a76a6b6)

##### Rozwiązanie

In [10]:
class TrojkatRownoboczny(Figura):
    def __init__(self, a):
        self.a = a
        self.h = 0.5 * a * math.sqrt(3)

    def obwod(self):
        super().obwod()
        return 3 * self.a

    def pole(self):
        super().pole()
        return 0.5 * self.a * self.h

In [11]:
f2 = TrojkatRownoboczny(5)

In [12]:
print(f2.obwod())

Obwód: 
15


In [13]:
print(f2.pole())

Pole: 
10.825317547305481


#### Trapez prostokątny

##### Ćwiczenie

Dane: podstawy `a`, `b` i wysokość `h`. Pole to średnia podstaw razy wysokość. Skośny bok oblicz z twierdzenia Pitagorasa.

![](https://upload.wikimedia.org/wikipedia/commons/1/11/Trapezoid.svg)

<img src="https://upload.wikimedia.org/wikipedia/commons/4/48/Trapezoid_right-angled.svg" alt="Trapez prostokątny: podstawy a i b, prostopadły bok d i skośny bok c" width="450">

$$h = d$$

![](https://wikimedia.org/api/rest_v1/media/math/render/svg/1a4014369cf2d0485beb70f64a4c50ff1ffbd482)

##### Rozwiązanie

In [14]:
class TrapezProstokatny(Figura):
    def __init__(self, a, b, h):
        self.a, self.b, self.h, self.d = a, b, h, h

        # czwarty bok (self.c) obliczamy z Pitagorasa
        e = b - a
        self.c = (h ** 2 + e ** 2) ** 0.5

    def obwod(self):
        super().obwod()
        return sum([self.a, self.b, self.c, self.d])

    def pole(self):
        super().pole()
        return 0.5 * (self.a + self.b) * self.h

In [15]:
f3 = TrapezProstokatny(2, 4, 3)

In [16]:
print(f3.obwod())

Obwód: 
12.60555127546399


In [17]:
print(f3.pole())

Pole: 
9.0


#### Równoległobok

##### Ćwiczenie

Dane: boki `a`, `b` i wysokość `h` do boku `a`. Oblicz pole i obwód.

![](https://upload.wikimedia.org/wikipedia/commons/b/be/Równoległobok.png)
![](https://wikimedia.org/api/rest_v1/media/math/render/svg/103619ee02e061d20aded35d00cce677d83777a8)
![](https://wikimedia.org/api/rest_v1/media/math/render/svg/407035bb1091fab5fd404263d116b30c9094ab90)

##### Rozwiązanie 1

In [18]:
class Rownoleglobok(Figura):
    def __init__(self, a, b, h):
        # inny sposób przypisywania zmiennych
        # trochę skraca liczbę linii
        self.a, self.b, self.h = a, b, h

    def obwod(self):
        super().obwod()
        return 2 * (self.a + self.b)

    def pole(self):
        super().pole()
        return self.a * self.h

In [19]:
f4 = Rownoleglobok(2, 4, 3)

In [20]:
print(f4.obwod())

Obwód: 
12


In [21]:
print(f4.pole())

Pole: 
6


##### Rozwiązanie 2: użycie tych samych wzorów, nie hierarchia typów

W tym wariancie podstawiamy do metod trapezu parami równe boki. Wynik obliczeń jest poprawny, ale równoległobok nie jest rodzajem trapezu prostokątnego. W dalszych zadaniach korzystamy z `Rownoleglobok` z rozwiązania 1.

In [ ]:
class RownoleglobokZeWzorowTrapezu(TrapezProstokatny):
    def __init__(self, a, b, h):
        self.a, self.b, self.c, self.d, self.h = a, a, b, b, h
        
        # Pokaz ponownego użycia wzorów, nie wzór hierarchii klas.
        # Równoległobok nie jest podtypem trapezu prostokątnego.

In [23]:
f4 = RownoleglobokZeWzorowTrapezu(2, 4, 3)

In [24]:
print(f4.obwod())

Obwód: 
12


In [25]:
print(f4.pole())

Pole: 
6.0


##### Prostokąt jako szczególny przypadek równoległoboku

Prostokąt z podstaw dziedziczy bezpośrednio po `Figura`. Tutaj użyjemy `Rownoleglobok` z rozwiązania 1, żeby odziedziczyć także wzory. Wysokość prostokąta jest równa jego drugiemu bokowi.

In [30]:
class ProstokatZRownolegloboku(Rownoleglobok):
    def __init__(self, a, b):
        super().__init__(a, b, b)

In [31]:
prostokat_z_rownolegloboku = ProstokatZRownolegloboku(2, 5)

In [32]:
print(prostokat_z_rownolegloboku.obwod())

Obwód: 
14


In [33]:
print(prostokat_z_rownolegloboku.pole())

Pole: 
10


### Klasa `Rectangle` (prostokąt) i dziedziczona (podrzędna) klasa `Parallelepipede` (równoległościan)

#### Ćwiczenie 1

Napisz **klasę** (**`class`**) **`Rectangle`** w języku Python, umożliwiając zbudowanie prostokąta z atrybutami **długości** (**`length`**) i **szerokości** (**`width`**).

#### Ćwiczenie 2

Utwórz metodę **`perimeter()`**, aby obliczyć obwód prostokąta i metodę **`area()`**, aby obliczyć powierzchnię prostokąta.

#### Ćwiczenie 3

Utwórz metodę **`display()`**, która wyświetla długość, szerokość, obwód i obszar obiektu utworzonego przy użyciu instancji klasy **`Rectangle`**.

#### Ćwiczenie 4

Utwórz klasę `Parallelepipede` z prostokątną podstawą i pionowymi krawędziami (prostopadłościan). Dodaj `height` i metodę `volume()`: pole podstawy razy wysokość.

Pierwszy wariant ma pokazać, jak działa dziedziczenie po `Rectangle`. Potem porównamy go z modelem, w którym bryła ma prostokątną podstawę, ale sama nie jest prostokątem.

#### Rozwiązanie 1: reużycie przez dziedziczenie

Kod dziedziczy `area()` i `perimeter()` dla podstawy. To pokaz działania mechanizmu, nie model relacji geometrycznej.

In [47]:
class Rectangle:
    # Zdefiniuj konstruktora z atrybutami: długość i szerokość 
    def __init__(self, length , width):
        self.length = length
        self.width = width
        
    # Utwórz metodę dla obwodu
    def perimeter(self):
        return 2*(self.length + self.width)
    
    # Utwórz metodę dla powierzchni
    def area(self):
        return self.length*self.width   
    
    # Utwórz metodę dla wyświetlania
    def display(self):
        print("Długość prostokąta to: ", self.length)
        print("Szerokość prostokąta to: ", self.width)
        print("Obwód prostokąta to: ", self.perimeter())
        print("Pole prostokąta to: ", self.area())
class Parallelepipede(Rectangle):
    def __init__(self, length, width, height):
        Rectangle.__init__(self, length, width)
        self.height = height
        
    # Zdefiniuj metodę dla objętości
    def volume(self):
        return self.area() * self.height

In [48]:
myRectangle = Rectangle(7, 5)
myRectangle.display()
print("----------------------------------")
myParallelepipede = Parallelepipede(7, 5, 2)
print("Objętość prostopadłościanu wynosi: ", myParallelepipede.volume())

Długość prostokąta to:  7
Szerokość prostokąta to:  5
Obwód prostokąta to:  24
Pole prostokąta to:  35
----------------------------------
Objętość prostopadłościanu wynosi:  70


#### Rozwiązanie 2: bryła ma prostokątną podstawę

Przechowujemy obiekt `Rectangle` i pytamy go o pole podstawy. W ten sposób samą bryłę modelujemy osobno.

In [ ]:
class Prostopadloscian:
    def __init__(self, podstawa, wysokosc):
        self.podstawa = podstawa
        self.wysokosc = wysokosc

    def volume(self):
        return self.podstawa.area() * self.wysokosc

In [ ]:
bryla = Prostopadloscian(Rectangle(7, 5), 2)
print(bryla.volume())

## Zadania utrwalające

#### Zadanie 1

Wróć do `KontoBankowe` i `KontoDebetowe` z tej lekcji. Sprawdź, czy nadal można wywołać odziedziczone `info()` i `wplac()` oraz przesłonięte `wyplac()`.

#### Zadanie 3

W klasach `A` i `B` zmień nazwy metod `fa()` i `fb()` na `f()`. Uruchom ponownie definicje klas, utwórz nowe `d = Pochodna()` i wywołaj `d.f()`.

Zamień kolejność rodziców w definicji `Pochodna`. Uruchom tę definicję, utwórz kolejne `d = Pochodna()` i ponownie wywołaj `d.f()`. Która metoda działa w każdej z wersji?